In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
import pandas as pd
from datetime import datetime
from io import BytesIO
from google.cloud import storage, bigquery
import numpy as np
import warnings
pd.set_option('display.max_columns', None)
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

In [ ]:
PROJECT_ID = "rs-nprd-dlk-agspc-roy-5b05"
BUCKET_NAME = "rs-nprd-dlk-ue4-gcs-ryl-sftp_generics"
FOLDER_PATH= "data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/20-agosto-2025/"
DATASET_ID = "produccion"
TABLE_ID= "ERRORES_desgravamen_prestamos"
FECHA_PERIODO= "2025-08-20"

#PROJECT_ID = "test-proyect-468615"
#BUCKET_NAME = "data_bucket_proy"
#FOLDER_PATH= "desgravamen_prestamos/"
#DATASET_ID = "db_test"
#TABLE_ID= "desgravamen_prestamos"


# SCRIPT COMPLETO

In [ ]:
### CLIENTES DE STORAGE Y BIGQUERY
storage_client = storage.Client(project=PROJECT_ID)
bigquery_client = bigquery.Client(project=PROJECT_ID)

bucket = storage_client.bucket(BUCKET_NAME)
blob_errores = bucket.blob('data_entries/REPORTES DE ERRORES/tabla_errores2.xlsx')
df_errores= pd.read_excel(BytesIO(blob_errores.download_as_string()), sheet_name='Sheet1', dtype={'CODIGO_ERROR': str, 'IDEERROR': str})
df_errores.loc[df_errores['CODIGO_ERROR'].isna(), 'CODIGO_ERROR'] = ""
#df_errores.drop(['IDEERROR'], axis=1, inplace=True)

schema_desgramen = [
        bigquery.SchemaField("NRO_LOTE", bigquery.enums.SqlTypeNames.INTEGER),
        bigquery.SchemaField("LOTES_ANTERIORES", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_CARGA", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("CODIGO_PRODUCTO", bigquery.enums.SqlTypeNames.INTEGER),
        bigquery.SchemaField("PRODUCTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CODIGO_PLAN", bigquery.enums.SqlTypeNames.INTEGER),
        bigquery.SchemaField("NOMBRE_DE_PLAN", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("COD_DE_CERTIFICADO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECINI_ALTA_CERTIFICADO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("FECFIN_ALTA_CERTIFICADO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("IDEDET", bigquery.enums.SqlTypeNames.INTEGER),
        bigquery.SchemaField("TIPO_MOVIMIENTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_INICIO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("FECHA_FIN ", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("MONEDA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("SUMA_ASEGURADA", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("TASA", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("TASA_RECARGO", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("PRIMABRUTACAN", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("PRIMANETACAN", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("NOMCOMPLETO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("APEPATERNO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("APEMATERNO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECNACIMIENTO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("TIPDOCUMENTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NUMDOCUMENTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NOMBRE_DE_ARCHIVO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("LINEA_TRAMA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("ORIGEN_ERROR", bigquery.enums.SqlTypeNames.STRING),
        #bigquery.SchemaField("IDEERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_TRAMA", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("FECHA_PERIODO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("TIPO_ERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CODIGO_ERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("DESCRIPCION_ERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("DETALLE", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("SOLUCION", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("SOLUCION_2", bigquery.enums.SqlTypeNames.STRING)
]

#### FUNCION PARA LIMPIAR Y TRANSFORMAR LA COLUMNA A UN FORMATO DE FECHA
def limpiar_fecha(columna):
    col = columna.astype(str) # Convertir a string
    col = col.str.strip()  # Eliminar espacios
    # Quitar decimales tipo '19880329.0'
    col = col.str.replace(r"\.0$", "", regex=True)
    col = col.str[:8] # Mantener solo los 8 primeros caracteres
    # Reemplazar valores inválidos (que no tengan 8 dígitos) por NaN
    col = col.where(col.str.match(r"^\d{8}$"), np.nan)
    return pd.to_datetime(col, format="%Y%m%d", errors="coerce")  # Convertir a fecha

#### FUNCION PARA EXTRAER LA FECHA DEL NOMBRE DE ARCHIVO
def extraer_fecha(fecha):
    try:
        fecha_str = fecha.split("_")[2]
        return datetime.strptime(fecha_str, "%Y%m%d").date()
    except Exception:
        return None  # En caso de error

#### FUNCION PARA CLASIFICAR EL REGISTRO
def clasificar(row):
    # Si lote no es nulo
    if str(row['LOTES_ANTERIORES']).strip() != "":
        return 'Regularizacion no Exitosa'

    # Si último número del nombre de archivo > 100
    try:
        ultimo_numero = int(row['NOMBRE_DE_ARCHIVO'].split("_")[-1][:-4])
        if ultimo_numero > 100:
            return 'Regularizacion no Exitosa'
    except:
        pass

    # Si ninguna condición se cumple
    return 'Mes corriente'

#### FUNCION PARA GUARDAR UN DATASET EN UNA TABLA DE BIGQUERY
def Guardar_en_BigQuery(data, dataset_id, table_id, schema):
    #bigquery_client = bigquery.Client()
    table_ref = bigquery_client.dataset(dataset_id).table(table_id)
    try:
        tabla = bigquery_client.get_table(table_ref)
        tabla_existe = True
    except:
        tabla_existe = False

    if not tabla_existe:
        # Crear la tabla si no existe
        tabla = bigquery.Table(table_ref, schema=schema)
        tabla = bigquery_client.create_table(tabla)
        print(f'----- Se ha creado la tabla {table_id} en el dataset {dataset_id} -----')

    # Agregar los registros de data a la tabla existente o recién creada
    job_config = bigquery.LoadJobConfig()
    job_config.write_disposition = bigquery.WriteDisposition.WRITE_APPEND if tabla_existe else bigquery.WriteDisposition.WRITE_TRUNCATE
    job = bigquery_client.load_table_from_dataframe(data, table_ref, job_config=job_config)
    job.result()
    #print(f'----- REGISTROS AGREGADOS CORRECTAMENTE EN: {table_id} -------')
    return


### LISTAR LOS ARCHIVOS QUE ESTAN DENTRO DEL BUCKET
bucket = storage_client.bucket(BUCKET_NAME)
blobs_excels = list(bucket.list_blobs(prefix=FOLDER_PATH))

### CICLO POR LA LISTA DE ARCHIVOS EXCEL
for blob in blobs_excels:
  if blob.name.endswith(".xlsx"):
    print(f" ------- CARGANDO ARCHIVO: {blob.name}")
    file = blob.download_as_string()
    df_desgravamen= pd.read_excel(BytesIO(file), sheet_name='Sheet1', dtype={'COD DE CERTIFICADO': str, 'DESCRIPCION ERROR': str,
                                                                                           'IDELOTE': str, 'CODIGO ERROR': str,
                                                                                           'SUMA ASEGURADA':str, 'TASA': str,
                                                                                           'TASA RECARGO': str, 'PRIMABRUTACAN':str, 'PRIMANETACAN': str})
    ###### LIMPIEZA Y TRANSFORMACIONES

    # Colocar _ en los espacio de los nombres de las columnas
    df_desgravamen.columns = (df_desgravamen.columns.str.strip()
                                                    .str.upper()  # opcional: todo en mayúsculas
                                                    .str.replace(r'[^A-Za-z0-9_]', '_', regex=True)  # reemplazar estapacios por _
                              )
    # Borrar Columnas repetidas

    df_desgravamen.drop(['IDELOTE','IDEDET_1','DESCRIPCION_ERROR'], axis=1, inplace=True)
    #df_desgravamen.drop(['IDEDET_1','ORIGEN_ERROR','UNNAMED__32'], axis=1, inplace=True)
    # Renombrar la columna
    #df_desgravamen= df_desgravamen.rename(columns={'IDELOTE': 'LINEA_TRAMA','CODIGO_ERROR':'ORIGEN_ERROR', 'DESCRIPCION_ERROR': 'CODIGO_ERROR'})
    df_desgravamen= df_desgravamen.rename(columns={'CODIGO_ERROR': 'IDEERROR'})

    # Imputar valores nulos
    df_desgravamen= df_desgravamen.fillna({'LOTES_ANTERIORES':'', 'MONEDA': 'SIN DATO', 'ORIGEN_ERROR': 'SIN DATO', 'TIPDOCUMENTO':'SIN DATO'})
    df_desgravamen.loc[df_desgravamen['MONEDA'] == 'nan', 'MONEDA'] = 'SIN DATO'
    df_desgravamen.loc[df_desgravamen['ORIGEN_ERROR'] == 'nan', 'ORIGEN_ERROR'] = 'SIN DATO'

    # Cambiar el sepador de decimales
    df_desgravamen['SUMA_ASEGURADA'] = df_desgravamen['SUMA_ASEGURADA'].str.replace(',', '.', regex=False)
    df_desgravamen['TASA'] = df_desgravamen['TASA'].str.replace(',', '.', regex=False)
    df_desgravamen['TASA_RECARGO'] = df_desgravamen['TASA_RECARGO'].str.replace(',', '.', regex=False)
    df_desgravamen['PRIMABRUTACAN'] = df_desgravamen['PRIMABRUTACAN'].str.replace(',', '.', regex=False)
    df_desgravamen['PRIMANETACAN'] = df_desgravamen['PRIMANETACAN'].str.replace(',', '.', regex=False)

    #Cambiar el tipo de datos de las columnas
    df_desgravamen["LOTES_ANTERIORES"] = df_desgravamen["LOTES_ANTERIORES"].astype(str)
    df_desgravamen['FECHA_CARGA'] = pd.to_datetime(df_desgravamen['FECHA_CARGA'], format="%d/%m/%Y %I:%M:%S %p", errors='coerce')
    df_desgravamen['CODIGO_PRODUCTO'] = df_desgravamen['CODIGO_PRODUCTO'].fillna(0).astype('int')
    df_desgravamen["PRODUCTO"] = df_desgravamen["PRODUCTO"].astype(str)
    df_desgravamen['CODIGO_PLAN'] = df_desgravamen['CODIGO_PLAN'].fillna(0).astype('int')
    df_desgravamen["NOMBRE_DE_PLAN"] = df_desgravamen["NOMBRE_DE_PLAN"].astype(str)
    df_desgravamen["COD_DE_CERTIFICADO"] = df_desgravamen["COD_DE_CERTIFICADO"].astype(str)
    df_desgravamen['FECINI_ALTA_CERTIFICADO'] = pd.to_datetime(df_desgravamen['FECINI_ALTA_CERTIFICADO'], format='%d/%m/%Y', errors='coerce')
    df_desgravamen['FECFIN_ALTA_CERTIFICADO'] = pd.to_datetime(df_desgravamen['FECFIN_ALTA_CERTIFICADO'], format='%d/%m/%Y', errors='coerce')
    df_desgravamen["TIPO_MOVIMIENTO"] = df_desgravamen["TIPO_MOVIMIENTO"].astype(str)
    df_desgravamen['FEC__INICIO'] = pd.to_datetime(df_desgravamen['FEC__INICIO'], format='%d/%m/%Y', errors='coerce')
    df_desgravamen['FEC__FIN'] = pd.to_datetime(df_desgravamen['FEC__FIN'], format='%d/%m/%Y', errors='coerce')
    df_desgravamen['MONEDA'] = df_desgravamen['MONEDA'].astype(str)
    df_desgravamen['SUMA_ASEGURADA'] = pd.to_numeric(df_desgravamen['SUMA_ASEGURADA'], errors="coerce")
    df_desgravamen['TASA'] = pd.to_numeric(df_desgravamen['TASA'], errors="coerce")
    df_desgravamen['TASA_RECARGO'] = pd.to_numeric(df_desgravamen['TASA_RECARGO'], errors="coerce")
    df_desgravamen['PRIMABRUTACAN'] = pd.to_numeric(df_desgravamen['PRIMABRUTACAN'], errors="coerce")
    df_desgravamen['PRIMANETACAN'] = pd.to_numeric(df_desgravamen['PRIMANETACAN'], errors="coerce")
    df_desgravamen['NOMCOMPLETO'] = df_desgravamen['NOMCOMPLETO'].astype(str)
    df_desgravamen['APEPATERNO'] = df_desgravamen['APEPATERNO'].astype(str)
    df_desgravamen['APEMATERNO'] = df_desgravamen['APEMATERNO'].astype(str)
    df_desgravamen["FECNACIMIENTO"] = limpiar_fecha(df_desgravamen["FECNACIMIENTO"])
    df_desgravamen["TIPDOCUMENTO"] = df_desgravamen["TIPDOCUMENTO"].astype(str)
    df_desgravamen['NUMDOCUMENTO'] = df_desgravamen['NUMDOCUMENTO'].astype(str)
    df_desgravamen['NOMBRE_DE_ARCHIVO'] = df_desgravamen['NOMBRE_DE_ARCHIVO'].astype(str)
    df_desgravamen['LINEA_TRAMA'] = df_desgravamen['LINEA_TRAMA'].astype(str)
    df_desgravamen['ORIGEN_ERROR'] = df_desgravamen['ORIGEN_ERROR'].astype(str)
    #df_desgravamen['CODIGO_ERROR'] = df_desgravamen['CODIGO_ERROR'].apply(lambda x: str(int(x)).zfill(4) if pd.notnull(x) else np.nan)
    #df_desgravamen['CODIGO_ERROR'] = df_desgravamen['CODIGO_ERROR'].apply(lambda x: str(int(x)).zfill(4) if pd.notnull(x) else "")
    df_desgravamen['IDEERROR'] = df_desgravamen['IDEERROR'].apply(lambda x: str(int(x)).zfill(4) if pd.notnull(x) else "")

    # Crear nuevas columnas cextrayendo la fecha del nombre de archivo
    df_desgravamen['FECHA_TRAMA'] = df_desgravamen['NOMBRE_DE_ARCHIVO'].apply(extraer_fecha)
    df_desgravamen['FECHA_TRAMA']= pd.to_datetime(df_desgravamen['FECHA_TRAMA'], errors='coerce')
    # Columna para diferenciar el periodo de reporte de errores
    df_desgravamen['FECHA_PERIODO']= pd.to_datetime(FECHA_PERIODO)
    # Crear columna evaluando condiciones en el contenido de otras columnas
    df_desgravamen['TIPO_ERROR']= df_desgravamen.apply(clasificar, axis=1)

    # Cruzar con la Tabla de errores
    #df_desgravamen= df_desgravamen.merge(df_errores, on='CODIGO_ERROR', how='left')
    df_desgravamen= df_desgravamen.merge(df_errores, on='IDEERROR', how='left')
    df_desgravamen.drop(['IDEERROR'], axis=1, inplace=True)

    df_desgravamen["DESCRIPCION_ERROR"] = df_desgravamen["DESCRIPCION_ERROR"].astype(str)
    df_desgravamen["DETALLE"] = df_desgravamen["DETALLE"].astype(str)
    df_desgravamen["SOLUCION"] = df_desgravamen["SOLUCION"].astype(str)
    df_desgravamen["SOLUCION_2"] = df_desgravamen["SOLUCION_2"].astype(str)

    # Guardar tabla en BigQuery
    Guardar_en_BigQuery(df_desgravamen, DATASET_ID, TABLE_ID, schema_desgramen)
    print(f"### EL ARCHIVO: {blob.name} SE HA GUARDADO CORRECTAMENTE EN BIGQUERY ###")



 ------- CARGANDO ARCHIVO: data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/20-agosto-2025/446 - Desgravamen - 01.xlsx
### EL ARCHIVO: data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/20-agosto-2025/446 - Desgravamen - 01.xlsx SE HA GUARDADO CORRECTAMENTE EN BIGQUERY ###
 ------- CARGANDO ARCHIVO: data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/20-agosto-2025/446 - Desgravamen - 02.xlsx
### EL ARCHIVO: data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/20-agosto-2025/446 - Desgravamen - 02.xlsx SE HA GUARDADO CORRECTAMENTE EN BIGQUERY ###
 ------- CARGANDO ARCHIVO: data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/20-agosto-2025/446 - Desgravamen - 03.xlsx
### EL ARCHIVO: data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/20-agosto-2025/446 - Desgravamen - 03.xlsx SE HA GUARDADO CORRECTAMENTE EN BIGQUERY ###
 ------- CARGANDO ARCHIVO: data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/20-agosto-2025/446 - Desgravamen - 04.xlsx
##

ValueError: Worksheet named 'Sheet1' not found

--------------

In [ ]:
storage_client = storage.Client(project=PROJECT_ID)
bigquery_client = bigquery.Client(project=PROJECT_ID)

In [ ]:
bucket = storage_client.bucket(BUCKET_NAME)
blob_errores = bucket.blob('data_entries/REPORTES DE ERRORES/tabla_errores2.xlsx')

In [ ]:
df_errores= pd.read_excel(BytesIO(blob_errores.download_as_string()), sheet_name='Sheet1', dtype={'CODIGO_ERROR': str, 'IDEERROR': str})
df_errores.loc[df_errores['CODIGO_ERROR'].isna(), 'CODIGO_ERROR'] = ""
#df_errores.drop(['IDEERROR'], axis=1, inplace=True)
df_errores.head(3)

,IDEERROR,CODIGO_ERROR,DESCRIPCION_ERROR,DETALLE,SOLUCION,SOLUCION_2
0,0974,1068,YA EXISTE UN PAGO CON LA MISMA FECHA DE INICIO...,YA EXISTE UN PAGO CON LA MISMA FECHA DE INICIO...,CONFIGURACION SAS,ANULAR MASIVAS
1,0005,0003,ERROR EN CARGA DE TRAMA,ERROR EN CARGA DE TRAMA,ANALISIS EMISOR,NaN
2,1158,1155,TRAMA REPETIDA O DUPLICADA,TRAMA REPETIDA O DUPLICADA,CONFIGURACION SAS,ANULAR MASIVAS


In [ ]:
bucket = storage_client.bucket(BUCKET_NAME)
blobs_excels = list(bucket.list_blobs(prefix=FOLDER_PATH))

In [ ]:
for blob in blobs_excels:
  if blob.name.endswith(".xlsx"):
    #file = blob.download_as_string()
    #excel_file = pd.ExcelFile(file)
    print(blob.name)
    #print(excel_file.sheet_names)

data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/20-agosto-2025/446 - Desgravamen - 01.xlsx
data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/20-agosto-2025/446 - Desgravamen - 02.xlsx
data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/20-agosto-2025/446 - Desgravamen - 03.xlsx
data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/20-agosto-2025/446 - Desgravamen - 04.xlsx
data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/20-agosto-2025/446 - Desgravamen - 05.xlsx
data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/20-agosto-2025/446 - Desgravamen - 06.xlsx
data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/20-agosto-2025/446 - Desgravamen - 07.xlsx
data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/20-agosto-2025/446 - Desgravamen - 08.xlsx
data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/20-agosto-2025/446 - Desgravamen - 09.xlsx
data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/20-agosto-2025/446 - Desgravamen 

In [ ]:
#blob_consuer = bucket.blob('desgravamen_prestamos/desgravamen - consuer.csv')
#file = blob_consuer.download_as_string()
#df_desgravamen= pd.read_csv(BytesIO(file), dtype={'COD DE CERTIFICADO': str, 'DESCRIPCION ERROR': str, 'IDELOTE': str, 'CODIGO ERROR': str})

In [ ]:
print(blobs_excels[12].name)
file = blobs_excels[12].download_as_string()
df_desgravamen= pd.read_excel(BytesIO(file), sheet_name='Sheet1', dtype={'COD DE CERTIFICADO': str, 'DESCRIPCION ERROR': str,
                                                                                           'IDELOTE': str, 'CODIGO ERROR': str,
                                                                                           'SUMA ASEGURADA':str, 'TASA': str,
                                                                                           'TASA RECARGO': str, 'PRIMABRUTACAN':str, 'PRIMANETACAN': str})


data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/20-agosto-2025/701 - Desgravamen Migracion consumer.xlsx


In [ ]:
df_desgravamen.head(3)

,NRO LOTE,LOTES ANTERIORES,FECHA CARGA,CODIGO PRODUCTO,PRODUCTO,CODIGO PLAN,NOMBRE DE PLAN,COD DE CERTIFICADO,fecini_alta_CERTIFICADO,fecfin_alta_CERTIFICADO,IDEDET,TIPO MOVIMIENTO,FEC. INICIO,FEC. FIN,MONEDA,SUMA ASEGURADA,TASA,TASA RECARGO,PRIMABRUTACAN,PRIMANETACAN,NOMCOMPLETO,APEPATERNO,APEMATERNO,FECNACIMIENTO,TIPDOCUMENTO,NUMDOCUMENTO,NOMBRE DE ARCHIVO,LINEA TRAMA,IDELOTE,IDEDET_1,ORIGEN ERROR,CODIGO ERROR,DESCRIPCION ERROR
0,142173,NaN,01/08/2018 06:05:50 PM,5780.0,BBVA Desgravamen Migracion CF,262176.0,Plan BBVA Desgravamen Vehiculos y Motos Migrac...,00110151814000181932,2018-01-30,2019-06-30,244557631,Renovacion,2018-03-31,30/04/2018,USD,0,0,0,9.45,9.17,NaN,NaN,NaN,NaN,NaN,NaN,20100130204_0169001_20180801_005.TXT,94100110151814000181932 01U...,142173,244557631,Error Rimac,1106,[**ERROR**]La suma para reasegurar es cero.[PO...
1,142173,NaN,01/08/2018 06:05:50 PM,5780.0,BBVA Desgravamen Migracion CF,262176.0,Plan BBVA Desgravamen Vehiculos y Motos Migrac...,00110151814000181932,2018-01-30,2019-06-30,244557631,Renovacion,2018-03-31,30/04/2018,USD,0,0,0,9.45,9.17,NaN,NaN,NaN,NaN,NaN,NaN,20100130204_0169001_20180801_005.TXT,94100110151814000181932 01U...,142173,244557631,Error Rimac,1106,[**ERROR**]La suma para reasegurar es cero.[PO...
2,142173,NaN,01/08/2018 06:05:50 PM,5780.0,BBVA Desgravamen Migracion CF,262176.0,Plan BBVA Desgravamen Vehiculos y Motos Migrac...,00110151814000181932,2018-01-30,2019-06-30,244557631,Renovacion,2018-03-31,30/04/2018,USD,0,0,0,9.45,9.17,NaN,NaN,NaN,NaN,NaN,NaN,20100130204_0169001_20180801_005.TXT,94100110151814000181932 01U...,142173,244557631,Error Rimac,1106,[**ERROR**]La suma para reasegurar es cero.[PO...


In [ ]:
df_desgravamen.columns = (
    df_desgravamen.columns
    .str.strip()  # quitar espacios al inicio/fin
    .str.upper()  # opcional: todo en mayúsculas
    .str.replace(r'[^A-Za-z0-9_]', '_', regex=True)  # reemplazar todo lo que no sea letra/número/_ por _
)
df_desgravamen.drop(['IDELOTE','IDEDET_1','DESCRIPCION_ERROR'], axis=1, inplace=True)
df_desgravamen= df_desgravamen.rename(columns={'CODIGO_ERROR': 'IDEERROR'})
#df_desgravamen.drop(['IDEDET_1','ORIGEN_ERROR','UNNAMED__32'], axis=1, inplace=True)
#df_desgravamen= df_desgravamen.rename(columns={'IDELOTE': 'LINEA_TRAMA','CODIGO_ERROR':'ORIGEN_ERROR', 'DESCRIPCION_ERROR': 'CODIGO_ERROR'})

In [ ]:
df_desgravamen= df_desgravamen.fillna({'LOTES_ANTERIORES':'', 'MONEDA': 'SIN DATO', 'ORIGEN_ERROR': 'SIN DATO', 'TIPDOCUMENTO':'SIN DATO'})
df_desgravamen.loc[df_desgravamen['MONEDA'] == 'nan', 'MONEDA'] = 'SIN DATO'
df_desgravamen.loc[df_desgravamen['ORIGEN_ERROR'] == 'nan', 'ORIGEN_ERROR'] = 'SIN DATO'

In [ ]:
df_desgravamen['SUMA_ASEGURADA'] = df_desgravamen['SUMA_ASEGURADA'].str.replace(',', '.', regex=False)
df_desgravamen['TASA'] = df_desgravamen['TASA'].str.replace(',', '.', regex=False)
df_desgravamen['TASA_RECARGO'] = df_desgravamen['TASA_RECARGO'].str.replace(',', '.', regex=False)
df_desgravamen['PRIMABRUTACAN'] = df_desgravamen['PRIMABRUTACAN'].str.replace(',', '.', regex=False)
df_desgravamen['PRIMANETACAN'] = df_desgravamen['PRIMANETACAN'].str.replace(',', '.', regex=False)

In [ ]:
df_desgravamen.head()

,NRO_LOTE,LOTES_ANTERIORES,FECHA_CARGA,CODIGO_PRODUCTO,PRODUCTO,CODIGO_PLAN,NOMBRE_DE_PLAN,COD_DE_CERTIFICADO,FECINI_ALTA_CERTIFICADO,FECFIN_ALTA_CERTIFICADO,IDEDET,TIPO_MOVIMIENTO,FEC__INICIO,FEC__FIN,MONEDA,SUMA_ASEGURADA,TASA,TASA_RECARGO,PRIMABRUTACAN,PRIMANETACAN,NOMCOMPLETO,APEPATERNO,APEMATERNO,FECNACIMIENTO,TIPDOCUMENTO,NUMDOCUMENTO,NOMBRE_DE_ARCHIVO,LINEA_TRAMA,ORIGEN_ERROR,IDEERROR
0,142173,,01/08/2018 06:05:50 PM,5780,BBVA Desgravamen Migracion CF,262176,Plan BBVA Desgravamen Vehiculos y Motos Migrac...,00110151814000181932,2018-01-30,2019-06-30,244557631.0,Renovacion,2018-03-31,30/04/2018,USD,0,0,0,9.45,9.17,NaN,NaN,NaN,NaN,SIN DATO,NaN,20100130204_0169001_20180801_005.TXT,94100110151814000181932 01U...,Error Rimac,1106
1,142173,,01/08/2018 06:05:50 PM,5780,BBVA Desgravamen Migracion CF,262176,Plan BBVA Desgravamen Vehiculos y Motos Migrac...,00110151814000181932,2018-01-30,2019-06-30,244557631.0,Renovacion,2018-03-31,30/04/2018,USD,0,0,0,9.45,9.17,NaN,NaN,NaN,NaN,SIN DATO,NaN,20100130204_0169001_20180801_005.TXT,94100110151814000181932 01U...,Error Rimac,1106
2,142173,,01/08/2018 06:05:50 PM,5780,BBVA Desgravamen Migracion CF,262176,Plan BBVA Desgravamen Vehiculos y Motos Migrac...,00110151814000181932,2018-01-30,2019-06-30,244557631.0,Renovacion,2018-03-31,30/04/2018,USD,0,0,0,9.45,9.17,NaN,NaN,NaN,NaN,SIN DATO,NaN,20100130204_0169001_20180801_005.TXT,94100110151814000181932 01U...,Error Rimac,1106
3,142173,,01/08/2018 06:05:50 PM,5780,BBVA Desgravamen Migracion CF,262176,Plan BBVA Desgravamen Vehiculos y Motos Migrac...,00110151814000181932,2018-01-30,2019-06-30,244557633.0,Renovacion,2018-05-31,30/06/2018,USD,0,0,0,9.45,9.17,NaN,NaN,NaN,NaN,SIN DATO,NaN,20100130204_0169001_20180801_005.TXT,94100110151814000181932 01U...,Error Rimac,1106
4,142173,,01/08/2018 06:05:50 PM,5780,BBVA Desgravamen Migracion CF,262176,Plan BBVA Desgravamen Vehiculos y Motos Migrac...,00110151814000181932,2018-01-30,2019-06-30,244557633.0,Renovacion,2018-05-31,30/06/2018,USD,0,0,0,9.45,9.17,NaN,NaN,NaN,NaN,SIN DATO,NaN,20100130204_0169001_20180801_005.TXT,94100110151814000181932 01U...,Error Rimac,1106


In [ ]:
def limpiar_fecha(columna):
    col = columna.astype(str) # Convertir a string
    col = col.str.strip()  # Eliminar espacios
    # Quitar decimales tipo '19880329.0'
    col = col.str.replace(r"\.0$", "", regex=True)
    col = col.str[:8] # Mantener solo los 8 primeros caracteres
    # Reemplazar valores inválidos (que no tengan 8 dígitos) por NaN
    col = col.where(col.str.match(r"^\d{8}$"), np.nan)
    return pd.to_datetime(col, format="%Y%m%d", errors="coerce")  # Convertir a fecha

In [ ]:
df_desgravamen["LOTES_ANTERIORES"] = df_desgravamen["LOTES_ANTERIORES"].astype(str)
df_desgravamen['FECHA_CARGA'] = pd.to_datetime(df_desgravamen['FECHA_CARGA'], format="%d/%m/%Y %I:%M:%S %p", errors='coerce')
df_desgravamen['CODIGO_PRODUCTO'] = df_desgravamen['CODIGO_PRODUCTO'].fillna(0).astype('int')
df_desgravamen["PRODUCTO"] = df_desgravamen["PRODUCTO"].astype(str)
df_desgravamen['CODIGO_PLAN'] = df_desgravamen['CODIGO_PLAN'].fillna(0).astype('int')
df_desgravamen["NOMBRE_DE_PLAN"] = df_desgravamen["NOMBRE_DE_PLAN"].astype(str)
df_desgravamen["COD_DE_CERTIFICADO"] = df_desgravamen["COD_DE_CERTIFICADO"].astype(str)
df_desgravamen['FECINI_ALTA_CERTIFICADO'] = pd.to_datetime(df_desgravamen['FECINI_ALTA_CERTIFICADO'], format='%d/%m/%Y', errors='coerce')
df_desgravamen['FECFIN_ALTA_CERTIFICADO'] = pd.to_datetime(df_desgravamen['FECFIN_ALTA_CERTIFICADO'], format='%d/%m/%Y', errors='coerce')
df_desgravamen["TIPO_MOVIMIENTO"] = df_desgravamen["TIPO_MOVIMIENTO"].astype(str)
df_desgravamen['FEC__INICIO'] = pd.to_datetime(df_desgravamen['FEC__INICIO'], format='%d/%m/%Y', errors='coerce')
df_desgravamen['FEC__FIN'] = pd.to_datetime(df_desgravamen['FEC__FIN'], format='%d/%m/%Y', errors='coerce')
df_desgravamen['MONEDA'] = df_desgravamen['MONEDA'].astype(str)
df_desgravamen['SUMA_ASEGURADA'] = pd.to_numeric(df_desgravamen['SUMA_ASEGURADA'], errors="coerce")
df_desgravamen['TASA'] = pd.to_numeric(df_desgravamen['TASA'], errors="coerce")
df_desgravamen['TASA_RECARGO'] = pd.to_numeric(df_desgravamen['TASA_RECARGO'], errors="coerce")
df_desgravamen['PRIMABRUTACAN'] = pd.to_numeric(df_desgravamen['PRIMABRUTACAN'], errors="coerce")
df_desgravamen['PRIMANETACAN'] = pd.to_numeric(df_desgravamen['PRIMANETACAN'], errors="coerce")
df_desgravamen['NOMCOMPLETO'] = df_desgravamen['NOMCOMPLETO'].astype(str)
df_desgravamen['APEPATERNO'] = df_desgravamen['APEPATERNO'].astype(str)
df_desgravamen['APEMATERNO'] = df_desgravamen['APEMATERNO'].astype(str)
df_desgravamen["FECNACIMIENTO"] = limpiar_fecha(df_desgravamen["FECNACIMIENTO"])
df_desgravamen["TIPDOCUMENTO"] = df_desgravamen["TIPDOCUMENTO"].astype(str)
df_desgravamen['NUMDOCUMENTO'] = df_desgravamen['NUMDOCUMENTO'].astype(str)
df_desgravamen['NOMBRE_DE_ARCHIVO'] = df_desgravamen['NOMBRE_DE_ARCHIVO'].astype(str)
df_desgravamen['LINEA_TRAMA'] = df_desgravamen['LINEA_TRAMA'].astype(str)
df_desgravamen['ORIGEN_ERROR'] = df_desgravamen['ORIGEN_ERROR'].astype(str)
df_desgravamen['IDEERROR'] = df_desgravamen['IDEERROR'].apply(lambda x: str(int(x)).zfill(4) if pd.notnull(x) else "")
#df_desgravamen['CODIGO_ERROR'] = df_desgravamen['CODIGO_ERROR'].apply(lambda x: str(int(x)).zfill(4) if pd.notnull(x) else np.nan)
#df_desgravamen['CODIGO_ERROR'] = df_desgravamen['CODIGO_ERROR'].apply(lambda x: str(int(x)).zfill(4) if pd.notnull(x) else "")

In [ ]:
df_desgravamen['CODIGO_PRODUCTO'].value_counts()

,count
CODIGO_PRODUCTO,
5780,8408
0,7


In [ ]:
#### FUNCION PARA EXTRAER LA FECHA DEL NOMBRE DE ARCHIVO
def extraer_fecha(fecha):
    try:
        fecha_str = fecha.split("_")[2]
        return datetime.strptime(fecha_str, "%Y%m%d").date()
    except Exception:
        return None  # En caso de error

#### FUNCION PARA CLASIFICAR EL REGISTRO
def clasificar(row):
    # Si lote no es nulo
    if str(row['LOTES_ANTERIORES']).strip() != "":
        return 'Regularizacion no Exitosa'

    # Si último número del nombre de archivo > 100
    try:
        ultimo_numero = int(row['NOMBRE_DE_ARCHIVO'].split("_")[-1][:-4])
        if ultimo_numero > 100:
            return 'Regularizacion no Exitosa'
    except:
        pass

    # Si ninguna condición se cumple
    return 'Mes corriente'

#### FUNCION PARA GUARDAR UN DATASET EN UNA TABLA DE BIGQUERY
def Guardar_en_BigQuery(data, dataset_id, table_id, schema):
    #bigquery_client = bigquery.Client()
    table_ref = bigquery_client.dataset(dataset_id).table(table_id)
    try:
        tabla = bigquery_client.get_table(table_ref)
        tabla_existe = True
    except:
        tabla_existe = False

    if not tabla_existe:
        # Crear la tabla si no existe
        tabla = bigquery.Table(table_ref, schema=schema)
        tabla = bigquery_client.create_table(tabla)
        print(f'----- Se ha creado la tabla {table_id} en el dataset {dataset_id} -----')

    # Agregar los registros de data a la tabla existente o recién creada
    job_config = bigquery.LoadJobConfig()
    job_config.write_disposition = bigquery.WriteDisposition.WRITE_APPEND if tabla_existe else bigquery.WriteDisposition.WRITE_TRUNCATE
    job = bigquery_client.load_table_from_dataframe(data, table_ref, job_config=job_config)
    job.result()
    print(f'----- REGISTROS AGREGADOS CORRECTAMENTE EN: {table_id} -------')
    return

In [ ]:
df_desgravamen= df_desgravamen[df_desgravamen['PRIMABRUTACAN']<20000]   #Eliminar las filas con valores atipicos muy altos
df_desgravamen['FECHA_TRAMA'] = df_desgravamen['NOMBRE_DE_ARCHIVO'].apply(extraer_fecha)    # Crear nuevas columnas cextrayendo la fecha del nombre de archivo
df_desgravamen['FECHA_TRAMA']= pd.to_datetime(df_desgravamen['FECHA_TRAMA'], errors='coerce')
df_desgravamen['FECHA_PERIODO']= pd.to_datetime(FECHA_PERIODO)
df_desgravamen['TIPO_ERROR']= df_desgravamen.apply(clasificar, axis=1)    # Crear colunma evaluando condiciones en el contenido de otras columnas

In [ ]:
df_desgravamen.head(3)

,NRO_LOTE,LOTES_ANTERIORES,FECHA_CARGA,CODIGO_PRODUCTO,PRODUCTO,CODIGO_PLAN,NOMBRE_DE_PLAN,COD_DE_CERTIFICADO,FECINI_ALTA_CERTIFICADO,FECFIN_ALTA_CERTIFICADO,IDEDET,TIPO_MOVIMIENTO,FEC__INICIO,FEC__FIN,MONEDA,SUMA_ASEGURADA,TASA,TASA_RECARGO,PRIMABRUTACAN,PRIMANETACAN,NOMCOMPLETO,APEPATERNO,APEMATERNO,FECNACIMIENTO,TIPDOCUMENTO,NUMDOCUMENTO,NOMBRE_DE_ARCHIVO,LINEA_TRAMA,ORIGEN_ERROR,IDEERROR,FECHA_TRAMA,FECHA_PERIODO,TIPO_ERROR
0,142173,,2018-08-01 18:05:50,5780,BBVA Desgravamen Migracion CF,262176,Plan BBVA Desgravamen Vehiculos y Motos Migrac...,00110151814000181932,2018-01-30,2019-06-30,244557631,Renovacion,2018-03-31,2018-04-30,USD,0.0,0.0,0.0,9.45,9.17,nan,nan,nan,NaT,SIN DATO,nan,20100130204_0169001_20180801_005.TXT,94100110151814000181932 01U...,Error Rimac,1106,2018-08-01,2025-08-20,Mes corriente
1,142173,,2018-08-01 18:05:50,5780,BBVA Desgravamen Migracion CF,262176,Plan BBVA Desgravamen Vehiculos y Motos Migrac...,00110151814000181932,2018-01-30,2019-06-30,244557631,Renovacion,2018-03-31,2018-04-30,USD,0.0,0.0,0.0,9.45,9.17,nan,nan,nan,NaT,SIN DATO,nan,20100130204_0169001_20180801_005.TXT,94100110151814000181932 01U...,Error Rimac,1106,2018-08-01,2025-08-20,Mes corriente
2,142173,,2018-08-01 18:05:50,5780,BBVA Desgravamen Migracion CF,262176,Plan BBVA Desgravamen Vehiculos y Motos Migrac...,00110151814000181932,2018-01-30,2019-06-30,244557631,Renovacion,2018-03-31,2018-04-30,USD,0.0,0.0,0.0,9.45,9.17,nan,nan,nan,NaT,SIN DATO,nan,20100130204_0169001_20180801_005.TXT,94100110151814000181932 01U...,Error Rimac,1106,2018-08-01,2025-08-20,Mes corriente


In [ ]:
df_desgravamen[df_desgravamen['CODIGO_ERROR']== ''].head(3)

,NRO_LOTE,LOTES_ANTERIORES,FECHA_CARGA,CODIGO_PRODUCTO,PRODUCTO,CODIGO_PLAN,NOMBRE_DE_PLAN,COD_DE_CERTIFICADO,FECINI_ALTA_CERTIFICADO,FECFIN_ALTA_CERTIFICADO,IDEDET,TIPO_MOVIMIENTO,FEC__INICIO,FEC__FIN,MONEDA,SUMA_ASEGURADA,TASA,TASA_RECARGO,PRIMABRUTACAN,PRIMANETACAN,NOMCOMPLETO,APEPATERNO,APEMATERNO,FECNACIMIENTO,TIPDOCUMENTO,NUMDOCUMENTO,NOMBRE_DE_ARCHIVO,LINEA_TRAMA,ORIGEN_ERROR,CODIGO_ERROR,FECHA_TRAMA,FECHA_PERIODO,TIPO_ERROR
9908,1059106,,NaT,3827,Desgravamen + Desempleo BBVA,89705,Plan Contifacil,00110814174001115325,NaT,NaT,634881630,Renovacion,2023-09-11,2023-10-11,SOL,0.0,0.0,0.0,1.23,1.19,nan,nan,nan,NaT,SIN DATO,nan,20100130204_0157001_20231001_004.TXT,901001108141740011153250011081413020268856501P...,SIN DATO,,2023-10-01,2025-06-16,Mes corriente
9909,1059106,,NaT,3827,Desgravamen + Desempleo BBVA,89705,Plan Contifacil,00110814174002537800,NaT,NaT,634881990,Renovacion,2023-09-05,2023-10-05,SOL,0.0,0.0,0.0,0.73,0.71,nan,nan,nan,NaT,SIN DATO,nan,20100130204_0157001_20231001_004.TXT,901001108141740025378000011081415022693377301P...,SIN DATO,,2023-10-01,2025-06-16,Mes corriente
9910,1059106,,NaT,3827,Desgravamen + Desempleo BBVA,89705,Plan Contifacil,00110250894000451350,NaT,NaT,634874333,Renovacion,2023-09-15,2023-10-15,SOL,0.0,0.0,0.0,5.78,5.61,nan,nan,nan,NaT,SIN DATO,nan,20100130204_0157001_20231001_004.TXT,901001102508940004513500011025082020054049201P...,SIN DATO,,2023-10-01,2025-06-16,Mes corriente


In [ ]:
df_desgravamen['TIPDOCUMENTO'].value_counts()

,count
TIPDOCUMENTO,
SIN DATO,8404
2.0,9


In [ ]:
df_desgravamen[df_desgravamen['TIPDOCUMENTO'].isnull()].head(3)

,NRO_LOTE,LOTES_ANTERIORES,FECHA_CARGA,CODIGO_PRODUCTO,PRODUCTO,CODIGO_PLAN,NOMBRE_DE_PLAN,COD_DE_CERTIFICADO,FECINI_ALTA_CERTIFICADO,FECFIN_ALTA_CERTIFICADO,IDEDET,TIPO_MOVIMIENTO,FEC__INICIO,FEC__FIN,MONEDA,SUMA_ASEGURADA,TASA,TASA_RECARGO,PRIMABRUTACAN,PRIMANETACAN,NOMCOMPLETO,APEPATERNO,APEMATERNO,FECNACIMIENTO,TIPDOCUMENTO,NUMDOCUMENTO,NOMBRE_DE_ARCHIVO,LINEA_TRAMA,ORIGEN_ERROR,CODIGO_ERROR,FECHA_TRAMA,FECHA_PERIODO,TIPO_ERROR


In [ ]:
df_desgravamen= df_desgravamen.merge(df_errores, on='IDEERROR', how='left')
df_desgravamen.drop(['IDEERROR'], axis=1, inplace=True)
#df_desgravamen= df_desgravamen.merge(df_errores, on='CODIGO_ERROR', how='left')
df_desgravamen.head(3)

,NRO_LOTE,LOTES_ANTERIORES,FECHA_CARGA,CODIGO_PRODUCTO,PRODUCTO,CODIGO_PLAN,NOMBRE_DE_PLAN,COD_DE_CERTIFICADO,FECINI_ALTA_CERTIFICADO,FECFIN_ALTA_CERTIFICADO,IDEDET,TIPO_MOVIMIENTO,FEC__INICIO,FEC__FIN,MONEDA,SUMA_ASEGURADA,TASA,TASA_RECARGO,PRIMABRUTACAN,PRIMANETACAN,NOMCOMPLETO,APEPATERNO,APEMATERNO,FECNACIMIENTO,TIPDOCUMENTO,NUMDOCUMENTO,NOMBRE_DE_ARCHIVO,LINEA_TRAMA,ORIGEN_ERROR,FECHA_TRAMA,FECHA_PERIODO,TIPO_ERROR,CODIGO_ERROR,DESCRIPCION_ERROR,DETALLE,SOLUCION,SOLUCION_2
0,142173,,2018-08-01 18:05:50,5780,BBVA Desgravamen Migracion CF,262176,Plan BBVA Desgravamen Vehiculos y Motos Migrac...,00110151814000181932,2018-01-30,2019-06-30,244557631,Renovacion,2018-03-31,2018-04-30,USD,0.0,0.0,0.0,9.45,9.17,nan,nan,nan,NaT,SIN DATO,nan,20100130204_0169001_20180801_005.TXT,94100110151814000181932 01U...,Error Rimac,2018-08-01,2025-08-20,Mes corriente,0177,VALIDACIONES CONSECUENCIA ACSEL E,VALIDACIONES CONSECUENCIA ACSEL E,CONFIGURACION ACSEL E,NaN
1,142173,,2018-08-01 18:05:50,5780,BBVA Desgravamen Migracion CF,262176,Plan BBVA Desgravamen Vehiculos y Motos Migrac...,00110151814000181932,2018-01-30,2019-06-30,244557631,Renovacion,2018-03-31,2018-04-30,USD,0.0,0.0,0.0,9.45,9.17,nan,nan,nan,NaT,SIN DATO,nan,20100130204_0169001_20180801_005.TXT,94100110151814000181932 01U...,Error Rimac,2018-08-01,2025-08-20,Mes corriente,0177,VALIDACIONES CONSECUENCIA ACSEL E,VALIDACIONES CONSECUENCIA ACSEL E,CONFIGURACION ACSEL E,NaN
2,142173,,2018-08-01 18:05:50,5780,BBVA Desgravamen Migracion CF,262176,Plan BBVA Desgravamen Vehiculos y Motos Migrac...,00110151814000181932,2018-01-30,2019-06-30,244557631,Renovacion,2018-03-31,2018-04-30,USD,0.0,0.0,0.0,9.45,9.17,nan,nan,nan,NaT,SIN DATO,nan,20100130204_0169001_20180801_005.TXT,94100110151814000181932 01U...,Error Rimac,2018-08-01,2025-08-20,Mes corriente,0177,VALIDACIONES CONSECUENCIA ACSEL E,VALIDACIONES CONSECUENCIA ACSEL E,CONFIGURACION ACSEL E,NaN


In [ ]:
df_desgravamen['DESCRIPCION_ERROR'].value_counts()

,count
DESCRIPCION_ERROR,
VALIDACIONES CONSECUENCIA ACSEL E,7836
ERROR EN ALTA,281
ERROR EN CARGA DE TRAMA,228
TRAMA REPETIDA O DUPLICADA,5
SIN POLIZA,5
YA EXISTE UN PAGO CON LA MISMA FECHA DE INICIO DE VIGENCIA,2
NI ÉXITO NI ERROR,1


In [ ]:
df_desgravamen["DESCRIPCION_ERROR"] = df_desgravamen["DESCRIPCION_ERROR"].astype(str)
df_desgravamen["DETALLE"] = df_desgravamen["DETALLE"].astype(str)
df_desgravamen["SOLUCION"] = df_desgravamen["SOLUCION"].astype(str)
df_desgravamen["SOLUCION_2"] = df_desgravamen["SOLUCION_2"].astype(str)

In [ ]:
df_desgravamen.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8413 entries, 0 to 8412
Data columns (total 37 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   NRO_LOTE                 8413 non-null   int64         
 1   LOTES_ANTERIORES         8413 non-null   object        
 2   FECHA_CARGA              8413 non-null   datetime64[ns]
 3   CODIGO_PRODUCTO          8413 non-null   int64         
 4   PRODUCTO                 8413 non-null   object        
 5   CODIGO_PLAN              8413 non-null   int64         
 6   NOMBRE_DE_PLAN           8413 non-null   object        
 7   COD_DE_CERTIFICADO       8413 non-null   object        
 8   FECINI_ALTA_CERTIFICADO  8365 non-null   datetime64[ns]
 9   FECFIN_ALTA_CERTIFICADO  8365 non-null   datetime64[ns]
 10  IDEDET                   8413 non-null   int64         
 11  TIPO_MOVIMIENTO          8413 non-null   object        
 12  FEC__INICIO              8413 non-

In [ ]:
TABLE_ID= "ERRORES_desgravamen_prestamos"
schema_desgramen = [
        bigquery.SchemaField("NRO_LOTE", bigquery.enums.SqlTypeNames.INTEGER),
        bigquery.SchemaField("LOTES_ANTERIORES", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_CARGA", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("CODIGO_PRODUCTO", bigquery.enums.SqlTypeNames.INTEGER),
        bigquery.SchemaField("PRODUCTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CODIGO_PLAN", bigquery.enums.SqlTypeNames.INTEGER),
        bigquery.SchemaField("NOMBRE_DE_PLAN", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("COD_DE_CERTIFICADO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECINI_ALTA_CERTIFICADO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("FECFIN_ALTA_CERTIFICADO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("IDEDET", bigquery.enums.SqlTypeNames.INTEGER),
        bigquery.SchemaField("TIPO_MOVIMIENTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_INICIO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("FECHA_FIN ", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("MONEDA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("SUMA_ASEGURADA", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("TASA", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("TASA_RECARGO", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("PRIMABRUTACAN", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("PRIMANETACAN", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("NOMCOMPLETO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("APEPATERNO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("APEMATERNO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECNACIMIENTO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("TIPDOCUMENTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NUMDOCUMENTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NOMBRE_DE_ARCHIVO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("LINEA_TRAMA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("ORIGEN_ERROR", bigquery.enums.SqlTypeNames.STRING),
        #bigquery.SchemaField("IDEERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_TRAMA", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("FECHA_PERIODO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("TIPO_ERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CODIGO_ERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("DESCRIPCION_ERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("DETALLE", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("SOLUCION", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("SOLUCION_2", bigquery.enums.SqlTypeNames.STRING)
    ]
Guardar_en_BigQuery(df_desgravamen, DATASET_ID, TABLE_ID, schema_desgramen)

----- REGISTROS AGREGADOS CORRECTAMENTE EN: ERRORES_desgravamen_prestamos -------
